# Hunt: low-and-slow password spray (T1110.003)

**Hypothesis.** One source IP tries a single password against many accounts, slowly enough that the alert on
100 failures in 5 minutes never fires. The hot index keeps 14 days; the Machine Data Lake keeps the rest.

## 1. Parameters

In [1]:
SUSPECT_IP = "198.51.100.22"            # from the hot-index triage
WINDOW_DAYS = 90
MIN_USERS = 40             # distinct accounts failed per IP per day
MAX_TRIES_PER_USER = 3     # low and slow: a few attempts per account at most
MIN_FAIL_RATIO = 0.8       # a spray fails almost every attempt

In [2]:
import pandas as pd
import lake
import show

show.note(lake.describe())

lake: live | raw demo_auth_raw | shared demo_auth_hunt_90d


## 2. Full history in the raw table

SPL2 over the Machine Data Lake raw table, which keeps what the hot index has aged out.

In [3]:
raw = lake.search_raw(SUSPECT_IP, days=120)
weekly = raw.set_index("_time").resample("W")["user"].size()
hot_from = pd.Timestamp.now(tz="UTC").floor("D") - pd.Timedelta(days=14)
show.weekly(weekly, hot_from, f"events per week from {SUSPECT_IP}")
show.result(f"*{len(raw):,}* events from *{SUSPECT_IP}* against "
            f"*{raw['user'].nunique()}* accounts, first seen *{raw['_time'].min():%b %d}*")
show.note(f"the 14-day index holds {(raw['_time'] >= hot_from).sum():,} of them")

                        events per week from 198.51.100.22                      
   ┌───────────────────────────────────────────────────────────────────────────┐
   │                                                                █████      │
300┤                                                                █████      │
   │                                                                █████      │
200┤                                                                █████      │
   │█████                                                           █████      │
100┤█████                                                           █████ █████│
   │█████                                                    █████  █████ █████│
  0┤█████                                                    █████  █████ █████│
   └──┬────────────┬───────────┬────────────┬────────────┬────────────┬────────┘
   Sep 13       Sep 27      Oct 11       Oct 25       Nov 08       Nov 22       
■ only in the lake   ■ also 

## 3. The shared 90-day table

The promoted analytics table, read over Open Sharing straight into pandas.

In [4]:
tbl = lake.load_shared()
tbl["day"] = tbl["_time"].dt.floor("D")
tbl["is_fail"] = tbl["outcome"].eq("fail")
reach = tbl[tbl["is_fail"]].groupby("src_ip")["user"].nunique()
show.ranked(reach.nlargest(6), SUSPECT_IP, "accounts with a failed sign-in, top source IPs")
show.result(f"*{len(tbl):,}* sign-ins from *{tbl['src_ip'].nunique()}* source IPs, "
            f"*{tbl['day'].min():%b %d}* to *{tbl['day'].max():%b %d}*")
show.note(f"{tbl.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")

accounts with a failed sign-in, top source IPs
198.51.100.22 █████████████████████████████████████████████████████████████████ 333
  203.0.113.7 █████████████████████████████ 150
   198.18.0.1 █ 1
  198.18.0.10 █ 1
 198.18.0.100 █ 1
 198.18.0.101 █ 1
104,198 sign-ins from 405 source IPs, Aug 26 to Nov 24
8.2 MB in memory


## 4. Backtest the detection

In [5]:
g = ["src_ip", "day"]
daily = tbl.groupby(g).agg(attempts=("user", "size"),
                           fails=("is_fail", "sum"))
failed = tbl[tbl["is_fail"]]
daily["users"] = failed.groupby(g)["user"].nunique()
daily["users"] = daily["users"].fillna(0)
daily["ratio"] = daily["fails"] / daily["attempts"]
daily["tries"] = daily["fails"] / daily["users"].clip(lower=1)
rule = ((daily["users"] >= MIN_USERS)
        & (daily["ratio"] >= MIN_FAIL_RATIO)
        & (daily["tries"] <= MAX_TRIES_PER_USER))
hits = daily[rule].reset_index()
flagged = hits.groupby("src_ip").agg(days=("day", "size"),
                                     first=("day", "min"),
                                     last=("day", "max"))
busy = daily[(daily["users"] >= MIN_USERS) & ~rule].reset_index()
show.rule_map(daily.reset_index(), MIN_USERS, MIN_FAIL_RATIO,
              flagged=flagged.index, skipped=busy["src_ip"].unique(),
              title=f"every source IP, every day for {WINDOW_DAYS} days")
show.result(f"the rule flags *{len(flagged)}* IP in {WINDOW_DAYS} days:")
show.table(flagged.reset_index().assign(first=flagged["first"].dt.strftime("%b %d").values,
                                        last=flagged["last"].dt.strftime("%b %d").values),
           mark=flagged.index)
five = failed.groupby(["src_ip", failed["_time"].dt.floor("5min")]).size()
show.note(f"old alert: the most failures any IP made in 5 minutes is {five.max()}; it fires at 100")
show.note(f"skipped {busy['src_ip'].nunique()} busy IP: failure ratio at most {busy['ratio'].max():.0%}")
days = pd.date_range(tbl["day"].min(), tbl["day"].max(), freq="D")
series = daily.xs(SUSPECT_IP, level="src_ip")["users"].reindex(days, fill_value=0)
show.daily(series, MIN_USERS, f"accounts failed per day by {SUSPECT_IP}")

                       every source IP, every day for 90 days                   
    ┌───────────────────────────────────────┬──────────────────────────────────┐
100%┤ ·                                     │  ●  ●  ●● ●●  ● ●                │
    │                                       │                                  │
 80%├───────────────────────────────────────┼──────────────────────────────────┤
    │ ·                                     │                                  │
 60%┤                                       │                                  │
    │ ·                                     │                                  │
 40%┤ ·                                     │                                  │
    │ ·                                     │                                  │
 20%┤ ·                                     │                                  │
    │    ●                                  │●  ● ●●●●●●●●●●●●●●●●●●  ●●●     ●│
  0%┤·                      

## 5. Threat intel and privileged accounts

In [6]:
intel = pd.read_csv("threat_intel.csv")
priv = pd.read_csv("privileged_users.csv")
ips = list(flagged.index)
show.table(intel[intel["indicator"].isin(ips)], mark=ips)
ok = tbl[tbl["src_ip"].isin(ips) & ~tbl["is_fail"]]
wins = ok[["_time", "src_ip", "user"]].merge(priv, on="user")
show.result(f"privileged accounts the suspect got into: *{len(wins)}*")
for time, ip, user, role in wins.itertuples(index=False):
    show.alert(f"*{user}* ({role}) signed in from {ip} on *{time:%b %d %H:%M}*")

    indicator          category first_reported         source
198.51.100.22 password spraying     2026-09-04 community feed
privileged accounts the suspect got into: 1
svc_backup (backup operator) signed in from 198.51.100.22 on Nov 22 03:12


## 6. The detection to save

In [7]:
spl = f"""index=demo_auth sourcetype="demo:auth:lake"
| eval failed=if(outcome="fail", 1, 0)
| stats count AS attempts, sum(failed) AS fails,
    dc(eval(if(failed=1, user, null()))) AS users BY src_ip
| eval ratio=fails/attempts, tries=fails/users
| where users>={MIN_USERS} AND ratio>={MIN_FAIL_RATIO}
    AND tries<={MAX_TRIES_PER_USER}"""
show.result("Daily saved search, ready for review:")
print(spl)

Daily saved search, ready for review:
index=demo_auth sourcetype="demo:auth:lake"
| eval failed=if(outcome="fail", 1, 0)
| stats count AS attempts, sum(failed) AS fails,
    dc(eval(if(failed=1, user, null()))) AS users BY src_ip
| eval ratio=fails/attempts, tries=fails/users
| where users>=40 AND ratio>=0.8
    AND tries<=3
